# HPC & SLURM for Machine Learning on Explorer

**Goal:** Learn how to run ML training jobs on Northeastern's **Explorer** HPC cluster using SLURM.  
**Time needed:** ~1 hour  
**Prerequisites:** Basic Python, familiarity with PyTorch (from Module 07/08)

**RC Documentation:**
- [Connecting to Explorer](https://rc-docs.northeastern.edu/en/latest/connectingtocluster/index.html)
- [Partitions](https://rc.northeastern.edu/partitions/)
- [GPU Overview](https://rc-docs.northeastern.edu/en/latest/gpus/gpuoverview.html)
- [Accessing GPUs](https://rc-docs.northeastern.edu/en/latest/gpus/accessinggpus.html)

---

## Why HPC?

Your laptop works fine for MNIST with a small network. But what happens when you need to:

- Train on **ImageNet** (14M images, 1000 classes)
- Fine-tune a **transformer** with billions of parameters
- Run a **hyperparameter sweep** across 50 configurations
- Train for **days** without tying up your personal machine

That's where HPC clusters come in. Northeastern's **Explorer** cluster gives you access to powerful GPUs (H200s, A100s, V100s), large memory, and fast storage - shared across researchers and students.

---

## 1. Getting Access to Explorer

Before you can use the cluster, you need an RC account.

### For Students in This Course

Your instructor submits a **classroom access form** to RC. Once processed, you'll get access to:
- The cluster itself (SSH and Open OnDemand)
- A `/courses/` directory for course materials
- The `courses` and `courses-gpu` partitions

You'll receive an email confirmation once your access is granted (typically within 24 hours of the request).

### For Research Students

If you're working with a PI/lab, your PI sponsors your access:
1. Submit a **"Research Computing Access Request"** via ServiceNow
2. Your PI approves the request
3. Access is granted within 24 hours

**Support:** rchelp@northeastern.edu

---

## 2. Login Nodes vs. Compute Nodes

This is the most important concept to understand before using the cluster.

![Explorer Cluster Architecture](images/cluster-architecture.svg)

### Login Node
- This is where you land after `ssh login.explorer.northeastern.edu`
- Use it for: editing files, submitting jobs, checking job status, transferring files
- **DO NOT** run training or any CPU/GPU-intensive work here
- Automated bots monitor login node usage. If you run heavy work, you'll get an email warning from RC staff.

### Compute Nodes
- This is where your actual training happens
- You access them through SLURM (`sbatch` for batch jobs, `srun` for interactive sessions)
- Come in different flavors: CPU-only, GPU, big memory
- Once you reserve resources, **use them or release them**. Idle allocations block other users.

---

## 3. Connecting to Explorer

### SSH Access

**Mac / Linux:**
```bash
ssh <your_neu_username>@login.explorer.northeastern.edu
```

**Windows:** Use **MobaXterm** (recommended by RC) or Windows Terminal:
```bash
ssh <your_neu_username>@login.explorer.northeastern.edu
```

### Web Access (Open OnDemand)

No SSH needed! Access Explorer through your browser via **Open OnDemand** at the RC website. This gives you a file browser, terminal, and the ability to launch Jupyter notebooks on compute nodes directly.

### Setting Up SSH Keys (do this once)

```bash
# On your local machine
ssh-keygen -t ed25519 -C "your_email@northeastern.edu"
ssh-copy-id <your_neu_username>@login.explorer.northeastern.edu
```

Now you can connect without typing your password every time.

---

## 4. Storage on Explorer

Explorer has several storage areas. Knowing which to use is important.

![Explorer Storage Layout](images/storage-layout.svg)

| Location | Path | Quota | Purge Policy | Purpose |
|----------|------|-------|-------------|--------|
| **Home** | `/home/<username>` | **~50 GB** | Never purged | Scripts, code, small config files |
| **Scratch** | `/scratch/<username>` | **~10 TB** | **Purged each semester** | Large datasets, job output, temporary files |
| **Course** | `/courses/<COURSE_NUM>.<TERM_CODE>/` | **1 TB** per course | Archived after course ends | Course materials and student work |

> **Note:** `/scratch` uses Lustre (fast parallel filesystem), ideal for I/O-heavy training jobs.  
> `/home` is on slower NFS storage - good for code, not for large data.

### Course Directory Structure

Each course on Explorer gets its own directory under `/courses/`, named by course number and term code:

```
/courses/CS6140.202630/       # Our course - Machine Learning
/courses/CS6120.202630/       # Natural Language Processing
/courses/CS6180.202630/     # Computer Vision, etc.
```

For this class (CS6140), the directory looks like:

```
/courses/CS6140.202630/
├── data/           # Course datasets (read-only for students)
├── shared/         # Shared resources (read-only for students)
├── staff/          # Instructor and TA directories
└── students/
    ├── smith.j/    # Your personal directory (only you can access)
    ├── doe.a/
    └── ...
```

### Important Storage Rules

- **Scratch is not backup.** Everything on `/scratch` gets purged each semester. Move important results to `/home` or `/courses` when jobs finish.
- **Don't fill up `$HOME`.** 50 GB might seem like a lot, but conda environments can eat through it fast. Keep data on scratch.
- **Check your usage:**
  ```bash
  # Must run from a compute node, not the login node
  srun -p short --pty bash
  check-quota /home/$USER
  ```

---

## 5. Environment Setup: Conda on Explorer

Before you can run ML code on the cluster, you need a Python environment with the right packages. Explorer uses **modules** and **Conda** for this.

### What Are Modules?

On your laptop, software is just "installed" - you run `python` and it works. On an HPC cluster, hundreds of users need different versions of the same software (Python 3.9 vs 3.11, CUDA 11.8 vs 12.1, etc.). Installing them all globally would create conflicts.

The **module system** solves this. Software is pre-installed but "hidden" by default. You selectively load what you need for your session:

![Module System](images/module-system.svg)

```bash
module load anaconda3/2024.06    # Now conda/python are available
module load cuda/12.1            # Now CUDA toolkit is available
```

What `module load` actually does under the hood is modify your shell environment variables - mainly `$PATH` and `$LD_LIBRARY_PATH`. So when you type `python`, the shell knows to look in the Anaconda 2024.06 directory instead of the system default.

Think of it like a toolbox shelf:

| Command | What it does |
|---------|-------------|
| `module avail` | What tools are on the shelf? |
| `module load X` | Pull tool X off the shelf onto your workbench |
| `module list` | What's currently on my workbench? |
| `module unload X` | Put it back on the shelf |

**This is the #1 thing that trips up new users.** If you SSH in and try `conda create ...` without first running `module load anaconda3/2024.06`, the command simply won't exist. Same with `nvcc` or any CUDA tools without `module load cuda/12.1`.

### Creating a Conda Environment

This is similar to the virtual environments we covered in [Module 00 - Python IDE Setup](../00_introduction/03-python.md), but Conda is more powerful. It manages not just Python packages but also system libraries, CUDA, and compiled code.

```bash
# First, get on a compute node (don't build envs on the login node)
srun --partition=short --nodes=1 --cpus-per-task=1 --pty /bin/bash

# Load Conda
module load anaconda3/2024.06

# Create an environment (store it in /home to persist across jobs)
conda create --prefix=/home/$USER/envs/ml101 python=3.11

# Activate it
conda activate /home/$USER/envs/ml101

# Install packages
conda install pytorch torchvision torchaudio pytorch-cuda=12.1 -c pytorch -c nvidia
conda install scikit-learn numpy pandas matplotlib

# Verify
python -c "import torch; print(f'PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}')"
```

### Using an `environment.yml` File

For reproducibility, define your environment in a YAML file:

```yaml
name: ml101
channels:
  - pytorch
  - nvidia
  - conda-forge
  - defaults
dependencies:
  - python=3.11
  - pytorch
  - torchvision
  - torchaudio
  - pytorch-cuda=12.1
  - scikit-learn
  - numpy
  - pandas
  - matplotlib
```

Then create the environment from it:

```bash
conda env create -f environment.yml --prefix=/home/$USER/envs/ml101
```

### Conda vs. venv (from Module 00)

In [Module 00](../00_introduction/03-python.md), we used `python -m venv` for virtual environments. Here's how they compare:

| Feature | `venv` (Module 00) | Conda (HPC) |
|---------|-------------------|-------------|
| Python packages | `pip install` | `conda install` |
| System libraries (CUDA, etc.) | Manual | Managed by Conda |
| Python version | Uses system Python | Can specify any version |
| Best for | Local development | HPC with GPU/CUDA dependencies |

### Important Conda Warning

**Do NOT auto-initialize conda in your `.bashrc`.** Having `conda init` in your shell config can interfere with other software on the cluster. Always load it manually with `module load anaconda3/2024.06`.

### Housekeeping

```bash
conda env list                    # List all your environments
conda list --prefix /path/to/env  # Packages in an environment
conda clean --all                 # Free up space from cached packages
```

---

## 6. Transferring Files

```bash
# Copy a file to Explorer
scp train_mnist.py <username>@login.explorer.northeastern.edu:/scratch/<username>/ml-101/

# Copy a whole directory
rsync -avz ./my_project/ <username>@login.explorer.northeastern.edu:/scratch/<username>/my_project/

# Copy results back to your laptop
scp <username>@login.explorer.northeastern.edu:/scratch/<username>/ml-101/output.log ./
```

**Tip:** `rsync` is smarter than `scp` - it only transfers files that changed, so it's great for syncing a project directory repeatedly.

You can also use **Open OnDemand's file browser** for drag-and-drop transfers.

---

## 7. SLURM - The Job Scheduler

You can't just run `python train.py` on the login node. Instead, you submit **jobs** through SLURM, and it schedules them on available compute nodes.

Think of it like a restaurant: you don't walk into the kitchen, you place an order and the kitchen handles it when there's capacity.

### Essential SLURM Commands

| Command | What it does | Example |
|---------|-------------|--------|
| `sinfo` | Show available partitions and nodes | `sinfo -s` |
| `squeue` | Show jobs in the queue | `squeue -u $USER` |
| `sbatch` | Submit a batch job | `sbatch train_job.sh` |
| `scancel` | Cancel a job | `scancel 12345` |
| `srun` | Run an interactive job | `srun --pty bash` |
| `sacct` | View past job history | `sacct --format=JobID,JobName,Elapsed,State` |
| `scontrol` | Show detailed job info | `scontrol show job 12345` |

Let's look at what each command does in practice. You would run these **on Explorer**, not in this notebook.

In [ ]:
# These are shell commands - run them on Explorer, not here!
# On the cluster, you'd type these directly in your terminal.

# -- Example output is shown in comments --

# Check what partitions are available
# $ sinfo -s
# PARTITION      AVAIL  TIMELIMIT   NODES(A/I/O/T)  NODELIST
# short*            up   2-00:00:0       ...
# gpu               up    8:00:00       ...
# gpu-short         up    2:00:00       ...
# gpu-interactive   up    2:00:00       ...
# courses           up   24:00:00       ...
# courses-gpu       up    8:00:00       ...

# Check GPU node availability
# $ sinfo -p gpu --Format=nodes,cpus,memory,features,statecompact,nodelist,gres

print("Run these commands on Explorer, not locally!")
print()
print("sinfo -s                                        # What partitions/nodes are available?")
print("sinfo -p gpu                                    # GPU partition details")
print("squeue -u $USER                                 # What jobs do I have running/pending?")
print("sacct --format=JobID,JobName,Elapsed,State      # My past jobs")
print("scontrol show job <JOBID>                       # Detailed info about a specific job")

---

## 8. CPU Nodes vs. GPU Nodes

Explorer has different types of compute nodes. Picking the right one matters for both performance and queue wait time.

![CPU vs GPU Nodes](images/cpu-vs-gpu.svg)

### CPU Partitions

| Partition | Default Time | Max Time | Use Case |
|-----------|-------------|----------|----------|
| `short` | 4 hours | 48 hours | Data preprocessing, feature engineering, scikit-learn |
| `courses` | 4 hours | 24 hours | Course-related CPU work (50 concurrent jobs) |

Use CPU nodes when your work doesn't benefit from GPU acceleration (data cleaning, classical ML algorithms, light scripting).

### GPU Partitions

| Partition | Default Time | Max Time | Max Jobs | GPU Limit | Best For |
|-----------|-------------|----------|---------|-----------|----------|
| `gpu` | 4 hours | 8 hours | 4 | 1 | Standard training jobs |
| `gpu-short` | 1 hour | 2 hours | 2 | 1 | Quick jobs (higher priority) |
| `gpu-interactive` | 1 hour | 2 hours | 2 | 1 | Debugging, interactive work |
| `courses-gpu` | 4 hours | 8 hours | 1 | 1 | Course-related GPU work |

Use GPU nodes when you're doing deep learning (PyTorch, TensorFlow). Neural network training is massively faster on GPUs.

### Available GPUs

Explorer has several GPU types across its nodes:

| GPU | VRAM | Nodes | SLURM `--gres` flag |
|-----|------|-------|--------------------|
| **H200** | 140 GB HBM3e | 4 nodes (8 GPUs each) | `--gres=gpu:h200:1` |
| **A100 (80GB)** | 80 GB HBM2e | 3 nodes | `--gres=gpu:a100:1` |
| **V100-SXM2** | 32 GB HBM2 | 17 nodes | `--gres=gpu:v100-sxm2:1` |
| **V100-PCIe** | 32 GB HBM2 | 4 nodes | `--gres=gpu:v100-pcie:1` |

**Important:** On the `gpu` partition, requesting more than 1 GPU will cause your job to fail.

**Tip:** You can request any available GPU with just `--gres=gpu:1`, but specifying the type (e.g., `gpu:v100-sxm2:1`) ensures you don't land on older hardware where your CUDA code might not work.

### When to Use What?

| Task | Partition | Why |
|------|-----------|-----|
| Preprocessing CSV data | `short` or `courses` | CPU-bound, no GPU needed |
| Training a neural network | `gpu` or `courses-gpu` | GPU accelerates matrix operations |
| Quick test (does my script run?) | `gpu-short` | Higher priority, faster scheduling |
| Interactive debugging with GPU | `gpu-interactive` | Get a live shell on a GPU node |

---

## 9. Anatomy of a SLURM Batch Script

A SLURM batch script is just a shell script with special `#SBATCH` directives at the top. These tell SLURM what resources your job needs.

```bash
#!/bin/bash
#SBATCH --job-name=mnist-train           # Name shown in squeue
#SBATCH --output=mnist_%j.out            # stdout log (%j = job ID)
#SBATCH --error=mnist_%j.err             # stderr log
#SBATCH --partition=gpu                  # Explorer GPU partition
#SBATCH --nodes=1                        # Single node
#SBATCH --ntasks=1                       # Single task
#SBATCH --gres=gpu:v100-sxm2:1           # 1x V100 SXM2 GPU
#SBATCH --cpus-per-task=4                # 4 CPU cores
#SBATCH --mem=4GB                        # 4 GB RAM
#SBATCH --time=04:00:00                  # Max runtime: 4 hours
#SBATCH --mail-type=END,FAIL             # Email when done or failed
#SBATCH --mail-user=you@northeastern.edu

# Load required modules
module load anaconda3/2024.06
module load cuda/12.1

# Activate your Conda environment
conda activate /home/$USER/envs/ml101

# Run your training script
python /scratch/$USER/ml-101/train_mnist.py --epochs 10 --lr 0.001
```

### Key `#SBATCH` flags explained

| Flag | What it controls | How to choose |
|------|-----------------|---------------|
| `--partition` | Which pool of nodes | `gpu` for training, `gpu-short` for quick tests |
| `--gres=gpu:TYPE:1` | GPU type and count | `v100-sxm2`, `a100`, `h200` - always `:1` on `gpu` partition |
| `--nodes` | Number of nodes | Always `1` for single-GPU jobs |
| `--ntasks` | Number of tasks | `1` for single-GPU training |
| `--cpus-per-task` | CPU cores | 4 is usually enough for data loading |
| `--mem` | RAM | 4-16 GB for most ML jobs |
| `--time` | Wall clock limit | Up to 8h on `gpu`, 2h on `gpu-short` |
| `--output` | Where stdout goes | Use `%j` for job ID to avoid overwrites |

---

## 10. Shell Scripts as Wrappers Around Python Code

In [Module 00](../00_introduction/03-python.md), we ran Python scripts directly: `python my_first_ml.py`. On the cluster, we wrap Python code in shell scripts for two reasons:

1. **SLURM directives** - the `#SBATCH` lines that request resources
2. **Environment setup** - loading modules and activating Conda before running Python

Think of the `.sh` script as the "setup and run" wrapper, and the `.py` script as the actual work.

![Wrapper Pattern](images/wrapper-pattern.svg)

### Example: Wrapping the Iris Classifier from Module 00

Remember [python-sample.py](../00_introduction/samples/python-sample.py) from Module 00? Here's how you'd run it on Explorer:

**The Python script** (`iris_classify.py`) - stays the same as your local version:
```python
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

def main():
    flower_data = load_iris()
    X, y = flower_data.data, flower_data.target
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

    model = RandomForestClassifier(random_state=42)
    model.fit(X_train, y_train)

    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)
    print(f"Model accuracy: {accuracy:.1%}")

if __name__ == "__main__":
    main()
```

**The shell wrapper** (`run_iris.sh`) - handles the HPC setup:
```bash
#!/bin/bash
#SBATCH --job-name=iris-classify
#SBATCH --output=iris_%j.out
#SBATCH --partition=short           # CPU-only, no GPU needed for scikit-learn
#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=1
#SBATCH --mem=2GB
#SBATCH --time=00:10:00             # 10 minutes is plenty

# Setup
module load anaconda3/2024.06
conda activate /home/$USER/envs/ml101

# Run
python iris_classify.py
```

```bash
$ sbatch run_iris.sh
Submitted batch job 123456
```

This separation keeps your Python code portable - it runs the same on your laptop and on the cluster. Only the shell wrapper changes.

---

## 11. Hands-On: Running MNIST on Explorer

Let's take the MNIST neural network from Module 08 and run it on the cluster. The process has 3 steps:

1. **Convert** the notebook to a standalone `.py` script
2. **Write** a SLURM batch script (the shell wrapper)
3. **Submit** and monitor the job

### Step 1: The Training Script

We've prepared `train_mnist.py` - it's the same model from Module 08, but packaged as a command-line script with argument parsing and logging. Let's look at the key pieces:

In [ ]:
# Let's examine the training script we'll submit to SLURM
# (This file lives alongside this notebook)

with open("train_mnist.py", "r") as f:
    print(f.read())

### Key differences from the notebook version:

1. **`argparse`** - command-line arguments for epochs, learning rate, batch size
2. **`logging`** - proper logging instead of `print()` (output goes to SLURM log files)
3. **Model saving** - saves the trained model to disk with `torch.save()`
4. **No matplotlib** - plotting doesn't work in a batch job (no display)
5. **`if __name__ == '__main__'`** - proper script entry point

### Step 2: The SLURM Batch Script

Let's look at the shell wrapper we'll use to submit this job:

In [ ]:
with open("submit_mnist.sh", "r") as f:
    print(f.read())

### Step 3: Submit and Monitor

On Explorer, you'd run:

```bash
# Submit the job
$ sbatch submit_mnist.sh
Submitted batch job 123456

# Check its status
$ squeue -u $USER
  JOBID  PARTITION  NAME         ST  TIME   NODES  NODELIST
  123456 gpu        mnist-train  R   0:05   1      gpu003

# Watch the output in real-time
$ tail -f mnist_123456.out
2026-04-01 10:00:01 - INFO - Using device: cuda
2026-04-01 10:00:01 - INFO - CUDA Device: Tesla V100-SXM2-32GB
2026-04-01 10:00:03 - INFO - Training set: 60000 samples
2026-04-01 10:00:03 - INFO - Test set: 10000 samples
2026-04-01 10:00:05 - INFO - Epoch 1/10
2026-04-01 10:00:08 - INFO - Train Loss: 0.4523, Train Acc: 86.2%
2026-04-01 10:00:09 - INFO - Test Loss: 0.1245, Test Acc: 96.1%
...

# Check GPU usage while the job runs
$ nvidia-smi    # (from within the compute node)

# Get detailed job info
$ scontrol show job 123456
```

---

## 12. Interactive Sessions

Sometimes you need to debug or experiment on a compute node directly. Use `srun` for this:

```bash
# Get an interactive shell on a GPU node (1 hour on gpu-interactive)
srun --partition=gpu-interactive --nodes=1 --gres=gpu:v100-sxm2:1 \
     --ntasks=1 --mem=4GB --time=01:00:00 --pty /bin/bash

# Now you're on a compute node - set up your environment
module load anaconda3/2024.06
module load cuda/12.1
conda activate /home/$USER/envs/ml101

# Verify GPU access
python -c "import torch; print(torch.cuda.is_available())"       # True!
python -c "import torch; print(torch.cuda.get_device_name(0))"   # Tesla V100-SXM2-32GB
nvidia-smi                                                        # GPU details
```

### Jupyter via Open OnDemand (easier)

Instead of SSH tunneling, Explorer's **Open OnDemand** portal lets you launch Jupyter notebooks on GPU nodes directly from your browser. This is the easiest way to do interactive GPU work.

### Jupyter via SSH Tunnel (manual)

```bash
# On the compute node, start Jupyter
jupyter lab --no-browser --port=8888

# On your local machine (separate terminal)
ssh -L 8888:gpu003:8888 <username>@login.explorer.northeastern.edu
# Then open http://localhost:8888 in your browser
```

---

## 13. Common Pitfalls and Tips

### Don'ts
- **Don't run training on the login node.** It's shared and your process will be killed. Bots monitor this and you'll get an email.
- **Don't store large datasets in `$HOME`.** Use `/scratch/$USER` - it has 10 TB vs. 50 GB.
- **Don't leave important data on `/scratch`.** It gets purged each semester. Move results to `/home` or `/courses` when done.
- **Don't underestimate `--time`.** If your job exceeds the time limit, SLURM kills it with no warning. Add a buffer.
- **Don't request more than 1 GPU on the `gpu` partition.** The job will simply fail.
- **Don't auto-initialize Conda in `.bashrc`.** It interferes with other cluster software. Load it manually with `module load`.
- **Don't leave idle interactive sessions running.** On `courses-gpu`, jobs idle for 1 hour get terminated automatically.

### Do's
- **Do save checkpoints during training.** If a job gets killed, you can resume from the last checkpoint.
- **Do use `sacct` to review past jobs.** Learn from how much time/memory your jobs actually used.
- **Do start with `gpu-short`.** Test your script with 1 epoch on `gpu-short` before committing to a long `gpu` job.
- **Do specify GPU type in `--gres`.** Avoids landing on older hardware where your CUDA code might fail.
- **Do use `conda clean --all` periodically.** Cached packages eat into your home directory quota.

In [ ]:
# Useful SLURM environment variables available inside your job
slurm_vars = {
    "SLURM_JOB_ID": "Unique job identifier (e.g., 123456)",
    "SLURM_JOB_NAME": "Job name from --job-name",
    "SLURM_NODELIST": "List of nodes assigned to your job",
    "SLURM_GPUS_ON_NODE": "Number of GPUs allocated",
    "SLURM_CPUS_PER_TASK": "Number of CPUs allocated",
    "SLURM_MEM_PER_NODE": "Memory allocated (in MB)",
    "SLURM_SUBMIT_DIR": "Directory where sbatch was called",
    "SLURM_TMPDIR": "Fast local scratch on the compute node",
}

print("SLURM Environment Variables (available inside your job)")
print("=" * 60)
for var, desc in slurm_vars.items():
    print(f"  ${var}")
    print(f"    -> {desc}")
    print()

---

## 14. Quick Reference: Your First Explorer Workflow

![Explorer Workflow](images/workflow.svg)

---

## Summary

| Concept | Key Takeaway |
|---------|-------------|
| **Explorer** | Northeastern's HPC cluster, accessed via `login.explorer.northeastern.edu` |
| **Login Node** | Where you SSH in - editing and job submission only, no heavy computation |
| **Compute Node** | Where your code actually runs - accessed through SLURM |
| **SLURM** | Job scheduler - you request resources, it runs your code |
| **`sbatch`** | Submit a batch job (most common) |
| **`srun`** | Start an interactive session for debugging |
| **`gpu` partition** | Up to 8 hours, 1 GPU per job (H200, A100, V100) |
| **`gpu-short`** | Up to 2 hours, higher priority - great for testing |
| **Conda** | Use `module load anaconda3/2024.06` to manage Python environments |
| **Shell wrapper** | `.sh` script handles SLURM + environment, `.py` script does the work |
| **Home** | `/home/$USER` - ~50 GB, for code and environments |
| **Scratch** | `/scratch/$USER` - ~10 TB, purged each semester, for data and output |
| **Course dir** | `/courses/<COURSE_NUM>.<TERM_CODE>/` - 1 TB, for class materials |
| **Open OnDemand** | Web portal for file management, Jupyter, and terminals |

### Next Steps

1. Make sure your RC account is set up
2. SSH into Explorer and explore the filesystem
3. Set up your Conda environment with PyTorch
4. Try submitting the MNIST training job from this demo
5. Start with `gpu-short` to test, then move to `gpu` for longer runs

---

## A Note on Using Powerful Tools Responsibly

HPC clusters, GPU nodes, AI-powered coding tools like Claude Code - these are incredibly powerful resources. They can compress weeks of work into hours and let you tackle problems that would otherwise be out of reach.

But power without understanding is a recipe for trouble.

### The Individual Cost

On an HPC cluster, a careless `rm -rf` on `/scratch` can wipe out months of experimental results. A misconfigured SLURM job can hog GPUs for hours while doing nothing useful, blocking your classmates. A training script with a bug in the data pipeline can burn through your entire GPU allocation producing garbage.

The same applies to AI coding assistants. Tools like Claude Code can generate and execute code at a speed that outpaces your ability to review it. If you don't understand what's being generated, you can't catch when it's wrong - and on a shared cluster, "wrong" has consequences beyond your own machine.

### The Bigger Picture

This isn't just about one bad SLURM job or one buggy script. There's a community-level cost when people operate tools they don't understand.

When students copy-paste generated code without reading it, they don't build the mental models they need. They can't debug when things break. They can't adapt when requirements change. They can't teach others. Over time, you end up with a generation of practitioners who can *operate* machine learning but can't *reason* about it - who can call `model.fit()` but can't explain why the loss isn't converging, who can request a GPU but can't tell if it's actually being used.

This creates a brittle ecosystem. Teams where nobody truly understands the training pipeline. Research groups that can't reproduce their own results. Engineers who stack abstractions on top of abstractions without anyone understanding what's underneath. When something breaks - and it always does - there's nobody left who can diagnose it from first principles.

The industry term for this is **"brain rot"** - and it's not about intelligence. It's about atrophy. If you never exercise the muscle of understanding what your code does, that muscle weakens. The tools get better, so you lean on them more, so you understand less, so you lean on them even more. It's a cycle, and it's already happening at scale across the ML community.

### What To Do About It

- **Use tools to go faster, not to skip learning.** Let AI help you write boilerplate, but make sure you can explain every line of the training loop.
- **Treat generated code like a colleague's PR.** Read it. Question it. Don't merge what you can't defend.
- **When something breaks, resist the urge to ask AI to fix it immediately.** Sit with the error. Read the traceback. Form a hypothesis. *Then* ask for help.
- **Build the habit of understanding one layer deeper.** If you're using PyTorch, understand what autograd does. If you're using SLURM, understand what a scheduler is. You don't need to know everything, but you need to know enough to ask the right questions.

The tools aren't the problem. The problem is outsourcing your understanding to them.

<div align="center">

![Monkey with a gun](https://i.imgflip.com/2/1bgw.jpg)

*Don't be this guy. Learn the tool before you pull the trigger.*

</div>